# 07 — human embryo clustering

**Feeds:** ED Fig 8a, 8b

**Position in the chain:** The trunk_main_dev copy: 46,055 cells, 48 clusters — the one ED Fig 8 reports.

Ported from the original analysis. Saved cell outputs are from the original run, and paths in them appear as `<analysis-root>/...`.

**Changes from the original notebook**, so that it runs from this repository:
1. Paths. The first code cell finds the repository from any working directory inside it; the original looked for `src/` at most two levels up. `PROJECT_ROOT`, which was the analysis directory, is now `scrnaseq/chain_inputs/`, the shipped files that no notebook writes, in the same layout. `DEV_ROOT`, under which the notebook writes, is `$SCRNASEQ_RESULTS_ROOT/trunk_main_dev/` (default `scrnaseq/output/trunk_main_dev/`) instead of the analysis directory's `trunk_main_dev/`. The earlier embryo SMD object, used only to compare gene lists, is therefore read from `scrnaseq/chain_inputs/legacy/results/intermediates/07_human_embryo/`.
2. The SMD z-score files are read from `scrnaseq/chain_inputs/trunk_main_dev/results/smd_runs/` (`PROJECT_ROOT / 'trunk_main_dev/results/smd_runs/...'`) instead of `RESULTS_DIR`, where they sat in the original analysis directory.
3. Removed the `.to_clipboard()` call, which copied a table to the system clipboard for pasting into the Supplementary Data spreadsheet and fail on a machine without a clipboard. The expression before each call is unchanged.
4. `w3-4_humanembryo.h5ad` (about 900 MB), which the original wrote into its working directory (the analysis directory's `trunk_main_dev/`), is written to `DEV_ROOT`, the same place under `$SCRNASEQ_RESULTS_ROOT`. No notebook reads it.

No other line of code was changed.


# 07 - Human Embryo Analysis

Depends on: `01_preprocessing` artifacts, local `trunk_main_dev/results/smd_runs/human_embryo__run*/` outputs, and legacy `07_human_embryo` intermediates for gene-list comparison only.

In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "trunk_morph_ref").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the repository root containing src/trunk_morph_ref/")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.trunk_morph_ref.paths import chain_inputs_root

# Inputs that no notebook writes: scrnaseq/chain_inputs/, laid out as the original analysis directory.
PROJECT_ROOT = chain_inputs_root(REPO_ROOT)

import os
os.chdir(REPO_ROOT)


## Setup and Imports


In [ ]:
import matplotlib

matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42

In [ ]:
import pickle
import random

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy
import seaborn as sns
from cmcrameri.cm import batlow
from scipy.sparse import coo_matrix, csr_matrix, load_npz, save_npz

/opt/anaconda3/lib/python3.12/site-packages/scanpy/_utils/__init__.py:27: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/opt/anaconda3/lib/python3.12/site-packages/anndata/__init__.py:70: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)
/opt/anaconda3/lib/python3.12/site-packages/anndata/__init__.py:70: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)
/opt/anaconda3/lib/python3.12/site-packages/anndata/__init__.py:70: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)
/opt/anaconda3/lib/python3.12/site-pa

In [ ]:
plt.rcParams["svg.fonttype"] = "none"  # Keep fonts editable in Illustrator

In [ ]:
from src.trunk_morph_ref.cell_ordering import ordered_cells_by_cluster_clustermap
from src.trunk_morph_ref.paths import init_output_paths, scrnaseq_results_root

# Outputs: $SCRNASEQ_RESULTS_ROOT (default scrnaseq/output/), under trunk_main_dev/ as in the original.
DEV_ROOT = scrnaseq_results_root(REPO_ROOT) / "trunk_main_dev"
RESULTS_DIR, MANUSCRIPT_FIG_DIR, EXTENDED_FIG_DIR = init_output_paths(DEV_ROOT)
from src.trunk_morph_ref.aggregation import (
    cluster_averages_sparse_safe,
)
from src.trunk_morph_ref.correlation import (
    gene_corrcoef_sparse_safe,
)
from src.trunk_morph_ref.plotting import (
    bold_selected_heatmap_yticklabels,
    hide_heatmap_axes_and_colorbar,
    keep_only_selected_heatmap_yticklabels,
    plot_empty_heatmap_axes,
)
from src.trunk_morph_ref.preprocessing import (
    assert_gene_id_index,
    contains_symbol,
    ensure_gene_id_index,
    filter_genes_sparse_safe as filter_genes,
    install_scanpy_symbol_defaults,
    normalize_unit_variance_sparse_safe as normalize_unit_variance,
    resolve_symbol_dict,
    resolve_symbols,
    symbols_missing,
    symbols_present,
    var_names_to_symbols,
)

# Use gene symbols for plot labels while keeping `var_names` as stable gene IDs.
install_scanpy_symbol_defaults(sc)


In [ ]:
from src.trunk_morph_ref.pipeline_io import (
    load_h5ad,
    load_npy,
    load_pickle,
    save_h5ad,
    save_json,
    save_npy,
    save_pickle,
    stage_dir,
)


In [ ]:
pre_path = stage_dir(RESULTS_DIR, '01_preprocessing')
adata_embryo_raw = load_h5ad(pre_path / 'adata_embryo_raw.h5ad')
adata_embryo = load_h5ad(pre_path / 'adata_embryo.h5ad')
adata_embryo_filtered = load_h5ad(pre_path / 'adata_embryo_filtered.h5ad')
adata_embryo_nofilter = load_h5ad(pre_path / 'adata_embryo_nofilter.h5ad')
for _adata in [
    adata_embryo_raw,
    adata_embryo,
    adata_embryo_filtered,
    adata_embryo_nofilter,
]:
    ensure_gene_id_index(_adata)
    assert_gene_id_index(_adata)

print('Loaded 01_preprocessing intermediates')


## Post-SMD processing and analysis
This section now selects a local `trunk_main_dev` embryo SMD run, compares it to a secondary local run for sensitivity, and compares the genes above the current z-score cutoff to the legacy embryo SMD gene list before clustering.

In [ ]:
# Load local embryo SMD runs and choose the promoted baseline run.
embryo_smd_run_paths = {
    1: PROJECT_ROOT / 'trunk_main_dev/results/smd_runs/human_embryo__run001/z_human_embryo_w3w4__smd_input_001.npy',
    2: PROJECT_ROOT / 'trunk_main_dev/results/smd_runs/human_embryo__run002/z_human_embryo_w3w4__smd_input_002.npy',
}
embryo_smd_run_n_sub = {
    1: 4000,
    2: 3750,
}
selected_embryo_smd_run = 1
comparison_embryo_smd_run = 2
embryo_z_threshold = 2.0

z_scores_by_run = {}
for run_id, run_path in embryo_smd_run_paths.items():
    z_scores = load_npy(run_path)
    if len(z_scores) != adata_embryo_filtered.n_vars:
        raise ValueError(
            f'{run_path.name} length ({len(z_scores)}) does not match '
            f'filtered embryo gene count ({adata_embryo_filtered.n_vars}).'
        )
    z_scores_by_run[run_id] = z_scores

z_scores_embryo = z_scores_by_run[selected_embryo_smd_run]
selected_embryo_smd_path = embryo_smd_run_paths[selected_embryo_smd_run]
comparison_embryo_smd_path = embryo_smd_run_paths[comparison_embryo_smd_run]

adata_embryo_filtered.var['z_score_SMD'] = z_scores_embryo
adata_embryo_filtered.var['log1p_z_score_SMD'] = np.log1p(z_scores_embryo)
adata_embryo_filtered.uns['smd_run_id'] = selected_embryo_smd_run
adata_embryo_filtered.uns['smd_run_n_sub'] = embryo_smd_run_n_sub[selected_embryo_smd_run]
adata_embryo_filtered.uns['smd_run_source_file'] = selected_embryo_smd_path.name

print(
    f'Selected embryo SMD run {selected_embryo_smd_run:03d} '
    f'(n_sub={embryo_smd_run_n_sub[selected_embryo_smd_run]}) '
    f'from {selected_embryo_smd_path}'
)


In [ ]:
# Plot top SMD z-scores for the selected embryo run.
with plt.rc_context({'figure.dpi': 150}):
    plt.figure(figsize=(12, 4))
    plt.hlines(embryo_z_threshold, -1, 2000, 'r')
    plt.plot(sorted(z_scores_embryo)[::-1], 'k.', markersize=5)
    plt.yscale('log')
    plt.ylim(0.01, 1.5 * z_scores_embryo.max())
    plt.xlim(-1, 1000)
    plt.ylabel('SMD z-score')
    plt.title(
        f'Embryo SMD run {selected_embryo_smd_run:03d} '
        f'(n_sub={embryo_smd_run_n_sub[selected_embryo_smd_run]})'
    )
    print()
    print(
        f'{(z_scores_embryo > embryo_z_threshold).sum()} '
        f'SMD genes with z_score > {embryo_z_threshold:g}'
    )
    plt.show()


In [ ]:
# Check that embryo SMD z-scores are insensitive to the nearby local run.
z_scores_embryo_comparison = z_scores_by_run[comparison_embryo_smd_run]
with plt.rc_context({'figure.dpi': 150}):
    plt.figure(figsize=(5, 5))
    plt.loglog(z_scores_embryo_comparison, z_scores_embryo, 'k.')
    plt.xlabel(
        f'run {comparison_embryo_smd_run:03d} '
        f'(n_sub={embryo_smd_run_n_sub[comparison_embryo_smd_run]}) z-scores'
    )
    plt.ylabel(
        f'run {selected_embryo_smd_run:03d} '
        f'(n_sub={embryo_smd_run_n_sub[selected_embryo_smd_run]}) z-scores'
    )
    plt.show()
print(scipy.stats.linregress(z_scores_embryo_comparison, z_scores_embryo))

current_gene_symbols = adata_embryo_filtered.var['gene_symbol'].astype(str).reset_index(drop=True)
selected_gene_symbols = set(current_gene_symbols[z_scores_embryo > embryo_z_threshold])
comparison_gene_symbols = set(
    current_gene_symbols[z_scores_embryo_comparison > embryo_z_threshold]
)
run_overlap_summary = pd.DataFrame(
    [
        {
            'selected_run': selected_embryo_smd_run,
            'selected_n_sub': embryo_smd_run_n_sub[selected_embryo_smd_run],
            'comparison_run': comparison_embryo_smd_run,
            'comparison_n_sub': embryo_smd_run_n_sub[comparison_embryo_smd_run],
            f'selected_z_gt_{embryo_z_threshold:g}': len(selected_gene_symbols),
            f'comparison_z_gt_{embryo_z_threshold:g}': len(comparison_gene_symbols),
            'intersection': len(selected_gene_symbols & comparison_gene_symbols),
            'union': len(selected_gene_symbols | comparison_gene_symbols),
            'jaccard': len(selected_gene_symbols & comparison_gene_symbols)
            / len(selected_gene_symbols | comparison_gene_symbols),
        }
    ]
)
display(run_overlap_summary)


### Legacy SMD Gene Comparison

In [ ]:
legacy_stage_path = PROJECT_ROOT / 'legacy/results/intermediates/07_human_embryo'
legacy_embryo_smd = load_h5ad(legacy_stage_path / 'adata_embryo_SMD.h5ad')
legacy_gene_symbols = pd.Index(legacy_embryo_smd.var_names.astype(str)).drop_duplicates()
legacy_gene_symbol_set = set(legacy_gene_symbols)

current_gene_table = pd.DataFrame(
    {
        'gene_id': adata_embryo_filtered.var_names.astype(str),
        'gene_symbol': adata_embryo_filtered.var['gene_symbol'].astype(str).to_numpy(),
        'gene_symbol_original': adata_embryo_filtered.var['gene_symbol_original']
        .astype(str)
        .to_numpy(),
        'selected_z': adata_embryo_filtered.var['z_score_SMD'].to_numpy(),
        'log1p_selected_z': adata_embryo_filtered.var['log1p_z_score_SMD'].to_numpy(),
    }
)
current_gene_table = current_gene_table.drop_duplicates(subset=['gene_id'], keep='first')
selected_gene_table = current_gene_table[
    current_gene_table['selected_z'] > embryo_z_threshold
].copy()
selected_gene_symbol_set = set(selected_gene_table['gene_symbol'])
intersection_gene_symbols = selected_gene_symbol_set & legacy_gene_symbol_set
union_gene_symbols = selected_gene_symbol_set | legacy_gene_symbol_set

legacy_comparison_summary = pd.DataFrame(
    [
        {
            'selected_run': selected_embryo_smd_run,
            'selected_n_sub': embryo_smd_run_n_sub[selected_embryo_smd_run],
            'legacy_gene_list_source': str(legacy_stage_path / 'adata_embryo_SMD.h5ad'),
            f'selected_z_gt_{embryo_z_threshold:g}': len(selected_gene_symbol_set),
            'legacy_gene_list_size': len(legacy_gene_symbol_set),
            'intersection': len(intersection_gene_symbols),
            'union': len(union_gene_symbols),
            'jaccard': len(intersection_gene_symbols) / len(union_gene_symbols),
            'legacy_recall': len(intersection_gene_symbols) / len(legacy_gene_symbol_set),
            'selected_precision_vs_legacy': len(intersection_gene_symbols)
            / len(selected_gene_symbol_set),
        }
    ]
)
display(legacy_comparison_summary)

symbol_lookup = (
    current_gene_table.sort_values(['gene_symbol', 'selected_z'], ascending=[True, False])
    .drop_duplicates(subset=['gene_symbol'], keep='first')
    .set_index('gene_symbol')
)

selected_only_genes = selected_gene_table[
    ~selected_gene_table['gene_symbol'].isin(legacy_gene_symbol_set)
][['gene_id', 'gene_symbol', 'gene_symbol_original', 'selected_z', 'log1p_selected_z']]
selected_only_genes = selected_only_genes.sort_values(
    ['selected_z', 'gene_symbol'], ascending=[False, True]
).reset_index(drop=True)

legacy_only_rows = []
for gene_symbol in sorted(legacy_gene_symbol_set - selected_gene_symbol_set):
    row = {'gene_symbol': gene_symbol}
    if gene_symbol in symbol_lookup.index:
        row['gene_id_on_current_filtered_space'] = symbol_lookup.loc[gene_symbol, 'gene_id']
        row['selected_z_on_current_filtered_space'] = float(
            symbol_lookup.loc[gene_symbol, 'selected_z']
        )
        row['log1p_selected_z_on_current_filtered_space'] = float(
            symbol_lookup.loc[gene_symbol, 'log1p_selected_z']
        )
    else:
        row['gene_id_on_current_filtered_space'] = np.nan
        row['selected_z_on_current_filtered_space'] = np.nan
        row['log1p_selected_z_on_current_filtered_space'] = np.nan
    legacy_only_rows.append(row)
legacy_only_genes = pd.DataFrame(legacy_only_rows)

print(f'Legacy-only genes at z > {embryo_z_threshold:g}: {len(legacy_only_genes)}')
display(legacy_only_genes)
print(f'Selected-run-only genes at z > {embryo_z_threshold:g}: {len(selected_only_genes)}')
display(selected_only_genes)


### Filter for top SMD genes by z-score

In [ ]:
# Filter for top embryo genes by the current z-score threshold.
adata_embryo_filtered_ = adata_embryo_filtered.copy()
adata_embryo_SMD = adata_embryo_filtered_[
    :, adata_embryo_filtered_.var.z_score_SMD > embryo_z_threshold
]
print(
    str(len(adata_embryo_SMD.var))
    + f' embryo genes selected with top SMD z-scores > {embryo_z_threshold:g}'
)


In [ ]:
corr_gg_embryo = gene_corrcoef_sparse_safe(adata_embryo_SMD.X)

cluster_gg_embryo = sns.clustermap(
    corr_gg_embryo,
    method="ward",
    metric="euclidean",
    figsize=(40, 40),
    cmap="viridis",
    vmin=-0.2,
    vmax=1,
    yticklabels=var_names_to_symbols(adata_embryo_SMD),
    xticklabels=var_names_to_symbols(adata_embryo_SMD),
)

### Manually add genes to gene set

In [ ]:
embryo_manual_smd_genes = [
    "TBXT",
    "MSGN1",
]
embryo_manual_remove_genes = []

In [ ]:


# Add manually curated marker genes after resolving symbols to gene IDs.
adata_embryo_SMD = ad.concat(
    [adata_embryo_SMD, adata_embryo[:, resolve_symbols(adata_embryo, embryo_manual_smd_genes, strict=False, allow_missing=True)]], axis=1, join="outer"
)

In [ ]:
# Resolve remove-list symbols to gene IDs so filtering uses the stable index.
embryo_manual_remove_ids = set(
    resolve_symbols(adata_embryo_SMD, embryo_manual_remove_genes, strict=False, allow_missing=True)
)
adata_embryo_SMD = adata_embryo_SMD[:, [g for g in adata_embryo_SMD.var_names if g not in embryo_manual_remove_ids]]

### Remove cell cycle genes

In [ ]:
genes_cellcycle = [
    "MKI67",
    "CENPE",
    "SGO2",
    "KIF14",
    "PIF1",
    "NDC80",
    "CDCA8",
    "PLK1",
    "AURKA",
    "UBE2C",
    "ASPM",
    "TOP2A",
    "TPX2",
    "NUSAP1",
    "CDC20",
    "CKS2",
    "KPNA2",
    "TUBB4B",
    "DLGAP5",
    "BIRC5",
    "HMMR",
    "CCNB1",
    "ARL6IP1",
    "PTTG1",
    "UBE2S",
    "DUT",
    "HELLS",
    "CLSPN",
    "RRM2",
    "PCLAF",
    "TYMS",
    "KIF18B",
    "SMC4",
    "HIST1H4C",
    "DIAPH3",
    "RFC3",
    "MIR924HG",
    "MIS18BP1",
    "TUBA1C",
    "CDK1",
    "CENPA",
    # extra genes added after mesoderm subclustering, were not already in SMD gene list
    "KIF11",
    "ECT2",
    "KNL1",
    "NEK2",
    "CEP55",
    "PSRC1",
    "CDCA3",
    "CCNA2",
    "GTSE1",
    "CENPF",
    "CKS1B",
    "MELK",
    "CDCA5",
    # Added 4/8 during LPM sub-subclustering
    "MCM4",
    "HIST1H1E",
    "BRCA2",
    "TRIM66",
    "C1orf112",
    "POLD3",
    "KIF23",
    # Added 4/8 part 2
    "CDKAL1",
    "RAD51AP1",
    "RANBP1",
    "CDCA2",
    "CAND2",
    "PCLAF",
    "EIF5A",
    "SRSF2",
    "GINS1",
    "GINS2",
    # Added 4/9 Neuron subclustering
    "NCAPG2",
    "CCND1",
    "ATAD2",
    "UBE2T",
    "CENPK",
    "LAPTM4B",
    "CCND2",
    "MED13",
    "ZNF37A",
    "PTPN14",
    # 4/10 Neural Crest
    "ncAPG",
    "TTK",
    "KIF4A",
    "KIF18A",
    "CDKN3",
    "CEP70",
    "BRIP1",
    "SPC25",
    "KIFC1",
    "NSD2",
    "BUB1",
    "BUB1B",
    "ANP32E",
    "HMGB3",
    "PCNA",
    "CENPP"
    # 4/10 Roof plate
    "CCT5",
    "KIF15",
    "CSRP2",
    "ORC6",
    "HMGB2",
]

In [ ]:
# Resolve cell-cycle symbols to gene IDs before dropping them from clustering features.
cellcycle_gene_ids = set(
    resolve_symbols(adata_embryo_SMD, genes_cellcycle, strict=False, allow_missing=True)
)
keep_genes_embryo = [
    i
    for i in range(len(adata_embryo_SMD.var_names))
    if adata_embryo_SMD.var_names[i] not in cellcycle_gene_ids
]
adata_embryo_SMD = adata_embryo_SMD[:, keep_genes_embryo]

corr_gg_embryo = gene_corrcoef_sparse_safe(adata_embryo_SMD.X)

cluster_gg_embryo = sns.clustermap(
    corr_gg_embryo,
    method="ward",
    metric="euclidean",
    figsize=(40, 40),
    cmap="viridis",
    vmin=-0.2,
    vmax=1,
    yticklabels=var_names_to_symbols(adata_embryo_SMD),
    xticklabels=var_names_to_symbols(adata_embryo_SMD),
)

### Supp. Data 2 - SMD Z-scores

In [ ]:
# For copy-paste into excel spreadsheet
# Supplementary Data 2, "Gene markers and differential expression supporting embryo cell type annotations and trunk morph correspondence"
# Sheet 2, "Embryo SMD Z-scores"
adata_embryo_SMD.var.z_score_SMD.sort_values(ascending=False)

## Clustering in SMD gene space

In [ ]:
# Create a copy and normalize to uniform reads among morph cells
adata_embryo_SMD_ = adata_embryo_SMD.copy()

sc.pp.normalize_total(adata_embryo_SMD_)
try:
    sc.pp.neighbors(adata_embryo_SMD_, use_rep="X")
except:
    sc.pp.neighbors(adata_embryo_SMD_, use_rep="X")
sc.tl.umap(adata_embryo_SMD_, random_state=0)

# Leiden clustering tuned to best align with legacy partition at default neighbors
sc.tl.leiden(
    adata_embryo_SMD_,
    resolution=2.98,
    random_state=0,
    key_added="leiden_embryo",
    flavor="igraph",
    n_iterations=-1,
)

In [ ]:
# We remove cluster 48 -- a forebrain doublet/multiplet cluster
adata_embryo_SMD_ = adata_embryo_SMD_[adata_embryo_SMD_.obs.leiden_embryo != "48", :]

In [ ]:
adata_embryo_with_clusters = adata_embryo.copy()
adata_embryo_with_clusters = adata_embryo_with_clusters[adata_embryo_SMD_.obs_names, :]
adata_embryo_with_clusters.obs["leiden_embryo"] = adata_embryo_SMD_.obs["leiden_embryo"]

In [ ]:
celltypes_embryo = {
    "20": "Week 4 Neuron - CNS Excitatory",  # ASCL1 NEUROG1 CRABP1+NOVA1 Midbrain?
    "38": "Week 4 Neuron - Peripheral Sensory",  # , NEUROD1 TLX3 POU4F1 SIX1 ISL2",
    "50": "Week 4 Neuron - Hypothalamus / Neuroendocrine",  # , Hypothalamus, Forebrain SIX6 POMC",
    "43": "Week 4 Neuron - CNS Inhibitory",  # LHX1 LHX5",
    "42": "Week 4 Neuron - Autonomic / Cholinergic",  # PHOX2A PHOX2B NKX6-1",
    "44": "Week 4 Neuron - Catecholaminergic / Glutamatergic",  # TFAP2B+ CRABP1+",
    "22": "Week 4 Head Mesenchyme - Multipotent Progenitors",  # No very specific markers",
    "23": "Week 4 Head Mesenchyme - Pharyngeal Arch Core Mesoderm",  # CYP1B1+ LPM",
    "47": "Week 4 Head Mesenchyme - Frontonasal Mesoderm",  # CCL2+ VEGFD LPM",
    "24": "Week 4 Head Mesenchyme - First Arch Oral / Palatal Mesenchyme",  #  SIX1+ LPM",
    "49": "Week 4 Head Mesenchyme - Branchiomeric Muscle Progenitors",  #  PITX2+ LPM",
    # "48": "Week 4 Head Mesenchyme - Forebrain-MULTIPLET",  #  48",
    "25": "Week 4 Forebrain - Optic Field",  # Eye field, RPE, SIX3, VSX2, RAX, MITF,
    "26": "Week 4 Forebrain - Diencephalon / Telencephalon",  # SOX2 EMX2 PAX6",
    "46": "Week 4 Dorsal Midbrain",  #  WNT2B OTX2",
    "45": "Week 4 Hindbrain",  # SOX2 PLP1 EDNRB POU3F2",
    "40": "Week 4 Roof Plate",  # WNT3A SOX2 PAX3 MSX1 OLIG3 ZIC1 MSX2 LMX1A
    "36": "Week 4 Forebrain / Midbrain",  #  SOX2 RAX POU3F1 DLK1
    "16": "Week 4 Dorsal Spinal Cord",  #  PAX3 SOX2
    "17": "Week 4 Intermediate-Ventral Spinal Cord",  #  HES5 SOX2 NKX6-2 OLIG2 SOX3
    "33": "Week 4 Neural Crest - Derivatives",  #  SOX10 FOXD3 TFAP2B
    "41": "Week 4 Neural Crest - Maturing Cranial",  #  CDH19 MCAM SOX10 FOXD3 TFAP2B
    "21": "Week 3 / 4 Notochord",  #  SHH TBXT
    "27": "Week 4 Definitive Endoderm - Fetal Liver",
    "37": "Week 4 Definitive Endoderm - Intestinal",
    "35": "Week 4 Intermediate Mesoderm / Kidney Progenitors",
    "11": "Week 4 Non-Neural Ectoderm - Surface Ectoderm",  #  TFAP2B+ WNT6 KRT7
    "15": "Week 4 Non-Neural Ectoderm - Cranial Placodal",  #  TFAP2B- SOX2+
    "28": "Week 4 Lateral Plate Mesoderm - Splanchnic",  # LHX2 GATA4 GATA6
    "18": "Week 4 Lateral Plate Mesoderm - Posterior",  #  PITX1 HAND1 HOXA10
    "19": "Week 4 Lateral Plate Mesoderm - Gut Visceral Mesenchyme",  #  HOXC8 HOXA10
    "30": "Week 4 Somite - Dorsal / Dermomyotome",  #  TCF15 PAX3 MEOX1 UNCX MEOX2 MEGF10
    "29": "Week 4 Lateral Plate Mesoderm - Craniofacial / Pharyngeal",  #  PRRX1 PRRX2 HAND2 TFAP2A SNAI1
    "31": "Week 4 Somite - Ventral / Sclerotome",  # FOXD1 TWIST1 MEOX2
    "32": "Week 4 Trunk Mesenchyme",  #  CYP1B1 FOXD1 LUM ZIC1 ZIC2 FLRT2
    "10": "Week 3 Non-Neural Ectoderm",  # SOX15+ KRT23 EPCAM
    "0": "Week 3 Intermediate Mesoderm",  # NPY OSR1 CDX4 CDX2 HAND1
    "7": "Week 3 Posterior Neural Tube / Neuromesodermal Progenitors",  #  TBXT CDX2 CDX4 HOXA7 HOXC6
    "8": "Week 3 Presomitic Mesoderm - Posterior",  #  MSGN1-
    "5": "Week 3 Early Somite",  # RIPPLY1 MEOX1 HOXA1
    "3": "Week 3 Lateral Plate Mesoderm - Anterior",  #  BMP4 HAND1 TNNT2 TMEM88
    "6": "Week 3 Lateral Plate Mesoderm - Craniofacial / Pharyngeal",  # PITX2 SIX1
    "2": "Week 3 Anterior Neuroectoderm",  #  SOX2 PAX6 CRABP1
    "9": "Week 3 Neural Crest - Early Migratory",  # SNAI2
    "4": "Week 3 / 4 Cardiomyocytes",  # MYL7
    "39": "Week 4 Skeletal Myocytes",
    "1": "Week 3 Endothelial",
    "34": "Week 4 Endothelial",  # SOX18
    "13": "Week 3 Presomitic Mesoderm - Anterior",  # MSGN1+
    "12": "Week 3 / 4 Erythroid Precursors",
    "14": "Week 3 / 4 Hematopoietic Progenitors",
}

adata_embryo_SMD_.obs["leiden_embryo"] = adata_embryo_SMD_.obs[
    "leiden_embryo"
].cat.rename_categories(celltypes_embryo)

adata_embryo_with_clusters.obs["leiden_embryo"] = adata_embryo_with_clusters.obs[
    "leiden_embryo"
].cat.rename_categories(celltypes_embryo)

celltypeorder_embryo = [
    # FB MB HB
    "Week 4 Forebrain - Optic Field",
    "Week 4 Forebrain / Midbrain",
    "Week 4 Forebrain - Diencephalon / Telencephalon",
    "Week 4 Dorsal Midbrain",
    "Week 4 Hindbrain",
    # Spinal Cord
    "Week 3 Anterior Neuroectoderm",
    # Dorsal NT
    "Week 4 Intermediate-Ventral Spinal Cord",
    "Week 4 Dorsal Spinal Cord",
    # Roof Plate
    "Week 4 Roof Plate",
    # Neural Crest
    "Week 3 Neural Crest - Early Migratory",
    "Week 4 Neural Crest - Derivatives",
    "Week 4 Neural Crest - Maturing Cranial",
    # Neuron
    "Week 4 Neuron - CNS Excitatory",  # ASCL1 NEUROG1 CRABP1+NOVA1 Midbrain?
    "Week 4 Neuron - Peripheral Sensory",  # , NEUROD1 TLX3 POU4F1 SIX1 ISL2",
    "Week 4 Neuron - Hypothalamus / Neuroendocrine",  # , Hypothalamus, Forebrain SIX6 POMC",
    "Week 4 Neuron - CNS Inhibitory",  # LHX1 LHX5",
    "Week 4 Neuron - Autonomic / Cholinergic",  # PHOX2A PHOX2B NKX6-1",
    "Week 4 Neuron - Catecholaminergic / Glutamatergic",  # TFAP2B+ CRABP1+",
    # NMPs
    "Week 3 Posterior Neural Tube / Neuromesodermal Progenitors",
    "Week 3 Presomitic Mesoderm - Posterior",
    # Notochord
    "Week 3 / 4 Notochord",
    # Presomitic mesoderm
    "Week 3 Presomitic Mesoderm - Anterior",
    # Early Somite
    "Week 3 Early Somite",
    # Somite
    "Week 4 Somite - Dorsal / Dermomyotome",
    "Week 4 Somite - Ventral / Sclerotome",
    # Intermediate MEsoderm
    "Week 4 Intermediate Mesoderm / Kidney Progenitors",
    # LPM
    "Week 3 Intermediate Mesoderm",  # Posterior Lateral / Intermediate Mesoderm, Newly Generated
    "Week 3 Lateral Plate Mesoderm - Anterior",  # Posterior Lateral Plate / Cardiac Mesoderm
    "Week 3 Lateral Plate Mesoderm - Craniofacial / Pharyngeal",  # Cranial / Pharyngeal Mesoderm
    "Week 4 Lateral Plate Mesoderm - Splanchnic",  # Splanchnic Mesoderm
    "Week 4 Lateral Plate Mesoderm - Craniofacial / Pharyngeal",  # Craniofacial/Pharyngeal Arch Mesoderm
    "Week 4 Lateral Plate Mesoderm - Posterior",  # Hindlimb /
    "Week 4 Lateral Plate Mesoderm - Gut Visceral Mesenchyme",
    "Week 3 / 4 Cardiomyocytes",
    "Week 4 Skeletal Myocytes",
    # Endothelial
    "Week 3 Endothelial",
    "Week 4 Endothelial",
    # Mesenchyme
    "Week 4 Head Mesenchyme - Multipotent Progenitors",
    # Broad stromal mesenchyme, extracellular matrix-rich, fibroblastic; contributes to non-somitic
    # craniofacial connective tissues. SIX2, MSX1, COL1A2, COL3A1, DCN, FRZB, ALCAM, LUM, RGCC
    "Week 4 Head Mesenchyme - Pharyngeal Arch Core Mesoderm",
    # Posterior/ventral arch mesenchyme, enriched for prechondrogenic and osteogenic genes; non-neural crest;
    # arch mesoderm-derived. FOXF2, CNMD, CTSK, DLX1/2, TBX15, LUM, VCAN, BGN, PITX1
    "Week 4 Head Mesenchyme - Frontonasal Mesoderm",
    # Midline craniofacial mesenchyme; contributes to nasal capsule and anterior skull base;
    # prechondrogenic/stromal. ALX1, ALX3, ALX4, ZIC1, MSX2, PRDM16, COL9A2, COL12A1, WIF1, CRYM
    "Week 4 Head Mesenchyme - First Arch Oral / Palatal Mesenchyme",  #
    # Arch 1 mesoderm-derived stromal mesenchyme patterned along oral and palatal axes; possibly dental/lingual
    # mesenchyme. DLX1, DLX2, LHX8, MAB21L1/2, WIF1, PRRX1, SPOCK3, TNC, SERTAD4
    "Week 4 Head Mesenchyme - Branchiomeric Muscle Progenitors",
    # Committed myogenic precursors from first and second arch mesoderm; give rise to facial, jaw, and extraocular
    # muscles. PITX2, MYF5, ALX1, ALX3, FOXC1, DKK2, RSPO1, EMX2, RARB, NEB
    # "Week 4 Head Mesenchyme - Forebrain-MULTIPLET",
    # Forebrain-adjacent stromal mesenchyme; expresses neuro-like regulators but mesoderm-derived; likely gives
    # rise to meninges and perivascular mesenchyme. FOXC1, PDGFRA, LUM, COL1A2, HES1, NR2F2, CDH2, ZIC1, FRZB
    "Week 4 Trunk Mesenchyme",
    # Non-neural Ectoderm
    "Week 4 Non-Neural Ectoderm - Surface Ectoderm",  # TFAP2A⁺, GRHL2⁺, TP63⁺, WNT6⁺
    "Week 4 Non-Neural Ectoderm - Cranial Placodal",  # DLX5⁺, HMX2⁺, IRX2⁺, FOXE1⁺
    "Week 3 Non-Neural Ectoderm",  # Generic epithelial, low patterning, SOX15⁺
    # Endoderm
    "Week 4 Definitive Endoderm - Fetal Liver",
    "Week 4 Definitive Endoderm - Intestinal",
    # Erythroid
    "Week 3 / 4 Erythroid Precursors",
    "Week 3 / 4 Hematopoietic Progenitors",
]

def reorder_embryo_categories(series, desired_order):
    current = list(series.cat.categories)
    ordered_present = [c for c in desired_order if c in current]
    extras = [c for c in current if c not in desired_order]
    if extras:
        print(
            "Embryo categories not present in manual order (appended at end): "
            + ", ".join(extras)
        )
    return series.cat.reorder_categories(ordered_present + extras)


adata_embryo_with_clusters.obs["leiden_embryo"] = reorder_embryo_categories(
    adata_embryo_with_clusters.obs["leiden_embryo"],
    celltypeorder_embryo,
)

adata_embryo_SMD_.obs["leiden_embryo"] = reorder_embryo_categories(
    adata_embryo_SMD_.obs["leiden_embryo"],
    celltypeorder_embryo,
)

adata_embryo_SMD_ = adata_embryo_SMD_[
    adata_embryo_SMD_.obs.sort_values("leiden_embryo").index, :
]
adata_embryo_with_clusters = adata_embryo_with_clusters[
    adata_embryo_with_clusters.obs.sort_values("leiden_embryo").index, :
]


In [ ]:
adata_embryo_SMD_leiden = sc.get.aggregate(
    adata_embryo_SMD_,
    by="leiden_embryo",
    func=["count_nonzero", "mean", "sum", "var"],
    axis="obs",
)

corr_clcl_embryo = np.corrcoef(adata_embryo_SMD_leiden.layers["mean"])

with plt.rc_context({"figure.figsize": (6, 5), "figure.dpi": (300)}):
    cluster_clcl_embryo = sns.clustermap(
        corr_clcl_embryo,
        method="ward",
        metric="euclidean",
        figsize=(10, 10),
        cmap=batlow,
        # vmin=-0.2,
        # vmax=1,
        yticklabels=adata_embryo_SMD_leiden.obs.leiden_embryo.cat.categories,
        xticklabels=adata_embryo_SMD_leiden.obs.leiden_embryo.cat.categories,
    )
    cluster_clcl_embryo.fig.suptitle(
        "Cluster-cluster mean gene correlation among all expressed genes"
    )

adata_embryo_SMD_ = adata_embryo_SMD_[
    adata_embryo_SMD_.obs.sort_values("leiden_embryo").index, :
]
adata_embryo_with_clusters = adata_embryo_with_clusters[adata_embryo_with_clusters.obs.sort_values("leiden_embryo").index, :]

In [ ]:
adata_embryo_with_clusters.var.index.name = "gene_index"
adata_embryo_with_clusters.write(DEV_ROOT / "w3-4_humanembryo.h5ad")

## ED Fig. 8a,b - Cluster visualizations: UMAP and stacked bar graphs

In [ ]:
with plt.rc_context({"figure.figsize": (6, 5), "figure.dpi": (300)}):
    sc.pl.umap(
        adata_embryo_SMD_,
        color="leiden_embryo",
        # size=5,
        title="Week 3 - 4 Human Embryos",
        show=False,
    )
    plt.savefig(
        f"{EXTENDED_FIG_DIR}/EDFig8a_umap_embryo_clusters.pdf",
        bbox_inches="tight",
        pad_inches=0,
    )

In [ ]:
adata_embryo_SMD_.obs["source"] = adata_embryo_with_clusters.obs["source"]

In [ ]:
with plt.rc_context({"figure.figsize": (6, 5), "figure.dpi": (300)}):
    sc.pl.umap(
        adata_embryo_SMD_,
        color="source",
        # size=5,
        title="Week 3 - 4 Human Embryos",
        show=False,
    )
    plt.savefig(
        f"{EXTENDED_FIG_DIR}/EDFig8a_umap_embryo_origin.pdf",
        bbox_inches="tight",
        pad_inches=0,
    )

In [ ]:
with plt.rc_context({"figure.figsize": (10, 5), "figure.dpi": (300)}):
    cluster_source_stackedbar_ax = pd.crosstab(
        adata_embryo_with_clusters.obs["source"],
        adata_embryo_with_clusters.obs["leiden_embryo"],
        normalize="columns",
    ).T.plot(kind="bar", stacked=True)
    cluster_source_stackedbar_ax.legend(title="Cluster ID", bbox_to_anchor=(1.7, 1.02), loc="upper right")
    plt.savefig(
        f"{EXTENDED_FIG_DIR}/EDFig8b_stackedbar_embryo_cluster_composition.pdf",
        bbox_inches="tight",
        pad_inches=0,
    )

## Save Stage Outputs
Persist human embryo clustering intermediates, plus an explicit full-gene embryo SMD z-score lookup table, for downstream notebooks and sensitivity workflows.


In [ ]:
stage_path = stage_dir(RESULTS_DIR, '07_human_embryo')
for _adata in [adata_embryo_with_clusters, adata_embryo_SMD_, adata_embryo_SMD]:
    assert_gene_id_index(_adata)
save_h5ad(adata_embryo_with_clusters, stage_path / 'adata_embryo_with_clusters.h5ad')
save_h5ad(adata_embryo_SMD_, stage_path / 'adata_embryo_SMD_.h5ad')
save_h5ad(adata_embryo_SMD, stage_path / 'adata_embryo_SMD.h5ad')

embryo_smd_gene_scores = pd.DataFrame(
    {
        'gene_ids': adata_embryo_filtered.var_names.astype(str),
        'gene_symbol': adata_embryo_filtered.var['gene_symbol'].astype(str).to_numpy(),
        'gene_symbol_original': adata_embryo_filtered.var['gene_symbol_original'].astype(str).to_numpy(),
        'z_score_SMD': adata_embryo_filtered.var['z_score_SMD'].to_numpy(),
        'log1p_z_score_SMD': adata_embryo_filtered.var['log1p_z_score_SMD'].to_numpy(),
        'selected_at_z_gt_2': (adata_embryo_filtered.var['z_score_SMD'].to_numpy() > 2),
        'selected_at_z_gt_current': (
            adata_embryo_filtered.var['z_score_SMD'].to_numpy() > embryo_z_threshold
        ),
        'smd_run_id': selected_embryo_smd_run,
        'smd_run_n_sub': embryo_smd_run_n_sub[selected_embryo_smd_run],
        'z_score_source_file': selected_embryo_smd_path.name,
    }
)
save_pickle(embryo_smd_gene_scores, stage_path / 'human_embryo_smd_gene_scores.pkl')
embryo_smd_gene_scores.to_csv(
    stage_path / 'human_embryo_smd_gene_scores.csv', index=False
)

save_pickle(legacy_comparison_summary, stage_path / 'human_embryo_legacy_gene_comparison_summary.pkl')
legacy_comparison_summary.to_csv(
    stage_path / 'human_embryo_legacy_gene_comparison_summary.csv', index=False
)
save_pickle(legacy_only_genes, stage_path / 'human_embryo_legacy_only_genes.pkl')
legacy_only_genes.to_csv(
    stage_path / 'human_embryo_legacy_only_genes.csv', index=False
)
save_pickle(selected_only_genes, stage_path / 'human_embryo_selected_only_genes.pkl')
selected_only_genes.to_csv(
    stage_path / 'human_embryo_selected_only_genes.csv', index=False
)

save_json(
    {
        'stage': '07_human_embryo',
        'seed_policy': 'all seeds set to 0',
        'selected_smd_run': int(selected_embryo_smd_run),
        'selected_smd_n_sub': int(embryo_smd_run_n_sub[selected_embryo_smd_run]),
        'selected_smd_source_file': selected_embryo_smd_path.name,
        'comparison_smd_run': int(comparison_embryo_smd_run),
        'comparison_smd_n_sub': int(embryo_smd_run_n_sub[comparison_embryo_smd_run]),
        'comparison_smd_source_file': comparison_embryo_smd_path.name,
        'smd_z_threshold_current': embryo_z_threshold,
        'legacy_gene_list_source': str(legacy_stage_path / 'adata_embryo_SMD.h5ad'),
    },
    stage_path / 'meta.json',
)
print(f'Saved embryo intermediates to {stage_path}')
